# Publications markdown generator for academicpages

Takes a TSV of publications with metadata and converts them for use with [academicpages.github.io](academicpages.github.io). This is an interactive Jupyter notebook ([see more info here](http://jupyter-notebook-beginner-guide.readthedocs.io/en/latest/what_is_jupyter.html)). The core python code is also in `publications.py`. Run either from the `markdown_generator` folder after replacing `publications.tsv` with one containing your data.

TODO: Make this work with BibTex and other databases of citations, rather than Stuart's non-standard TSV format and citation style.


## Data format

The TSV needs to have the following columns: pub_date, title, venue, excerpt, citation, site_url, and paper_url, with a header at the top. 

- `excerpt` and `paper_url` can be blank, but the others must have values. 
- `pub_date` must be formatted as YYYY-MM-DD.
- `url_slug` will be the descriptive part of the .md file and the permalink URL for the page about the paper. The .md file will be `YYYY-MM-DD-[url_slug].md` and the permalink will be `https://[yourdomain]/publications/YYYY-MM-DD-[url_slug]`

This is how the raw file looks (it doesn't look pretty, use a spreadsheet or other program to edit and create).

In [71]:
!cat publications.tsv

pub_date	title	venue	excerpt	citation	url_slug	paper_url
01/01/16	FRET analysis using sperm-activating peptides tagged with fluorescent proteins reveals that ligand-binding sites exist as clusters	Journal of Experimental Biology	Long-range cellular communication between the sperm and egg is critical for external fertilization. Sperm-activating peptides (SAPs) are diffusible components of the outer layer of eggs in echinoderms," and function as chemoattractants for spermatozoa. The decapeptide named speract is the best-characterized sea urchin SAP. Biochemical and physiological actions of speract have been studied with purified or chemically synthesized peptides. In this work"," we prepared recombinant speract fused to a fluorescent protein (FP; FP-speract) using three color variants: a cyan (eCFP)"," a yellow (mVenus) and a large Stokes shift yellow (mAmetrine) FP. Although these fluorescence tags are 20 times larger than speract"," competitive binding experiments using mAmetrine-spera

## Import pandas

We are using the very handy pandas library for dataframes.

In [72]:
import pandas as pd

## Import TSV

Pandas makes this easy with the read_csv function. We are using a TSV, so we specify the separator as a tab, or `\t`.

I found it important to put this data in a tab-separated values format, because there are a lot of commas in this kind of data and comma-separated values can get messed up. However, you can modify the import statement, as pandas also has read_excel(), read_json(), and others.

In [73]:
publications = pd.read_csv("publications.tsv", sep="\t", header=0)
publications


,pub_date,title,venue,excerpt,citation,url_slug,paper_url
0,01/01/16,FRET analysis using sperm-activating peptides ...,Journal of Experimental Biology,Long-range cellular communication between the ...,"Arcos-Hernández <i>et al."",</i> 2016",Arcos_2016,http://fraroco.github.io/files/ArcosC_JEB_2016...
1,01/07/17,FRET-based binding assay between a fluorescent...,FEBS Letters,The cyclic nucleotide-binding domain (CNBD) fu...,"Romero <i>et al."",</i> 2017",Romero_2017,http://fraroco.github.io/files/FRomero_FEBSLET...
2,02/07/14,Intracellular pH in sperm physiology,Biochemical and Biophysical Research Communica...,Intracellular pH (pHi) regulation is essential...,"Nishigaki <i>et al."","" </i> 2014",Nishigaki_2014,http://fraroco.github.io/files/NishigakiT_BIC_...
3,01/07/16,"CatSper,"" el canal de Ca2+ que regula el batid...",Revista Iberoamericana de Ciencia,CatSper is a sperm-specific Ca2+ channel. In m...,"Nishigaki <i>et al."","" </i> 2016",Nishigaki_2016,http://fraroco.github.io/files/NishigakiT_RIC_...
4,01/09/18,Robust evaluation of intermolecular FRET using...,BioTechniques,Fluorescence (or Förster) resonance energy tra...,"Santana-Calvo <i>et al."",</i> 2018",Santana_2018,http://fraroco.github.io/files/Santana-Calvo_B...
5,01/08/19,High potassium seawater inhibits ascidian sper...,Zygote,Male gamete chemotaxis towards the female game...,"Kinoshita-Terauchi <i>et al."","" </i> 2019",Kinoshita-Terauch_2019,http://fraroco.github.io/files/kinoshita-terau...
6,01/12/19,Comparative genomic analysis suggests that the...,Zoological Letters,CatSper is a sperm-specific calcium ion (Ca2+)...,"Romero <i>et al."",</i> 2019",Romero_2019,http://fraroco.github.io/files/romero_2020.pdf
7,01/01/23,How to study a highly toxic protein to bacteri...,Protein Expression and Purification,Heterologous expression systems have been used...,"Arcos <i>et al."",</i> 2023.",Arcos_2023,http://fraroco.github.io/files/romero_2020.pdf


## Escape special characters

YAML is very picky about how it takes a valid string, so we are replacing single and double quotes (and ampersands) with their HTML encoded equivilents. This makes them look not so readable in raw format, but they are parsed and rendered nicely.

In [74]:
html_escape_table = {
    "&": "&amp;",
    '"': "&quot;",
    "'": "&apos;"
    }

def html_escape(text):
    """Produce entities within text."""
    return "".join(html_escape_table.get(c,c) for c in text)

## Creating the markdown files

This is where the heavy lifting is done. This loops through all the rows in the TSV dataframe, then starts to concatentate a big string (```md```) that contains the markdown for each type. It does the YAML metadata first, then does the description for the individual page.

In [75]:
import os
for row, item in publications.iterrows():
    
    md_filename = str(item.pub_date) + "-" + item.url_slug + ".md"
    html_filename = str(item.pub_date) + "-" + item.url_slug
    year = item.pub_date[:4]
    
    ## YAML variables
    
    md = "---\ntitle: \""   + item.title + '"\n'
    
    md += """collection: publications"""
    
    md += """\npermalink: /publication/""" + html_filename
    
    if len(str(item.excerpt)) > 5:
        md += "\nexcerpt: '" + html_escape(item.excerpt) + "'"
    
    md += "\ndate: " + str(item.pub_date) 
    
    md += "\nvenue: '" + html_escape(item.venue) + "'"
    
    if len(str(item.paper_url)) > 5:
        md += "\npaperurl: '" + item.paper_url + "'"
    
    md += "\ncitation: '" + html_escape(item.citation) + "'"
    
    md += "\n---"
    
    ## Markdown description for individual page
        
    if len(str(item.excerpt)) > 5:
        md += "\n" + html_escape(item.excerpt) + "\n"
    
    if len(str(item.paper_url)) > 5:
        md += "\n[Download paper here](" + item.paper_url + ")\n" 
        
    md += "\nRecommended citation: " + item.citation
    
    md_filename = os.path.basename(md_filename)
       
    with open("../_publications/" + md_filename, 'w') as f:
        f.write(md)

These files are in the publications directory, one directory below where we're working from.

In [76]:
!ls ../_publications/

14-Nishigaki_2014.md         18-Santana_2018.md
16-Arcos_2016.md             19-Kinoshita-Terauch_2019.md
16-Nishigaki_2016.md         19-Romero_2019.md
17-Romero_2017.md            23-Arcos_2023.md


In [77]:
!cat ../_publications/17-Romero_2017.md


---
title: "FRET-based binding assay between a fluorescent cAMP analogue and a cyclic nucleotide-binding domain tagged with a CFP"
collection: publications
permalink: /publication/01/07/17-Romero_2017
excerpt: 'The cyclic nucleotide-binding domain (CNBD) functions as a regulatory domain of many proteins involved in cyclic nucleotide signalling. We developed a straightforward and reliable binding assay based on intermolecular fluorescence resonance energy transfer (FRET) between an adenosine-3’,5’-cyclic monophosphate analogue labelled with fluorescein and a recombinant CNBD of human EPAC1 tagged with a cyan fluorescence protein (CFP). The high FRET efficiency of this method (~ 80%) allowed us to perform several types of binding experiments with nanomolar range of sample using conventional equipment. In addition,&quot; the CFP tag on the CNBD enabled us to perform a specific binding experiment using an unpurified protein. Considering these advantages&quot;,&quot; this technique is usefu